In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
import torch

from src import config
from src.console import print_header, print_kv, print_status, print_subheader
from src.model_analysis import load_run_gate_weights, plot_gate_distribution, summarize_gate_values
from src.preprocessing import (extract_segmental_features_cached, extract_suprasegmental_features_cached,
                               mfcc_frame_count)
from src.splits import summarize_severity_loso_splits
from src.training.data import build_speaker_label_map, load_manifest
from src.training.models import MODEL_DESCRIPTIONS, SEVERITY_MODEL_NAME, build_model, parameter_counts
from src.training.reporting import (check_frozen_config_guard, print_feature_audit,
                                    print_final_run_configuration, write_frozen_config)
from src.training.runner import TrainingConfig, run_training
from src.training.utils import resolve_device, set_seed

config.ensure_directories()
df_m6 = load_manifest()

print_header("Three-Branch Gated-Fusion Severity Architecture -- Training")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))

In [ ]:
RUN_NAME = "severity_gated_fusion_three_branch"

cfg = TrainingConfig(
    task="severity",
    model=SEVERITY_MODEL_NAME,
    run_name=RUN_NAME,
    severity_protocol=config.SEVERITY_PRIMARY_PROTOCOL,
)

final_config = print_final_run_configuration(cfg, df_m6)

In [ ]:
print_subheader("Severity dataset -- 15 dysarthric speakers")

speaker_severity = (df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)]
                    .drop_duplicates("Speaker_ID")[["Speaker_ID", "Severity"]])
speaker_counts = speaker_severity["Severity"].value_counts().reindex(config.SEVERITY_CLASS_NAMES)
print_kv("Speakers per severity class", dict(speaker_counts))

utterance_counts = (df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)]
                    .groupby("Severity").size().reindex(config.SEVERITY_CLASS_NAMES))
print_kv("Utterances per severity class", dict(utterance_counts))
speaker_severity.sort_values(["Severity", "Speaker_ID"]).reset_index(drop=True)

In [ ]:
summarize_severity_loso_splits(df_m6)

In [ ]:
device = resolve_device(None)
set_seed(cfg.seed)

# A representative speaker-label map (excludes one held-out speaker) just to size the model for the checks...
example_train_df = df_m6[df_m6["Speaker_ID"] != config.DYSARTHRIC_IDS[0]]
example_speaker_map = build_speaker_label_map(example_train_df)

model = build_model(cfg.model, config.NUM_CLASSES[cfg.task],
                    num_speakers=len(example_speaker_map)).to(device)

print_kv("Model", MODEL_DESCRIPTIONS[cfg.model])
print_kv("Device", device)

In [ ]:
counts = parameter_counts(model)
lora_params = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "lora_" in n)

print_kv("Trainable parameters", f"{counts['trainable_params']:,}")
print_kv("Total parameters", f"{counts['total_params']:,}")
print_kv("Trainable %", f"{counts['trainable_pct']:.2f}%")
print_kv("Frozen parameters", f"{counts['total_params'] - counts['trainable_params']:,}")
print_kv("LoRA adapter parameters", f"{lora_params:,}")

In [ ]:
feature_audit_result = print_feature_audit(model=model, num_classes=config.NUM_CLASSES[cfg.task])

assert feature_audit_result["learned_branch"]["dimensions"] == config.LEARNED_EMBED_DIM == 128
assert feature_audit_result["segmental_branch"]["dimensions"] == config.SEGMENTAL_EMBED_DIM == 64
assert feature_audit_result["suprasegmental_branch"]["dimensions"] == config.SUPRA_EMBED_DIM == 64
assert feature_audit_result["fusion"]["fused_dim"] == config.FUSED_EMBED_DIM == 256
print_status("Branch dimensions match the frozen architecture spec exactly (128 / 64 / 64 -> 256)", ok=True)

In [ ]:
# COMPUTE BUDGET, STEP 0 -- measured batch-size selection (RTX 4060, 8GB).
from src.training.budget import benchmark_batch_sizes

batch_bench = benchmark_batch_sizes(df_m6, task="severity", model_name=SEVERITY_MODEL_NAME,
                                    batch_sizes=[16, 24, 32], epochs=1)
batch_bench

In [ ]:
# COMPUTE BUDGET, STEP 1 -- real per-fold-epoch cost, projected against the hard wall-clock cap for the...
from src.training.budget import ExperimentBudgetManager
from src.preprocessing import extract_segmental_features_cached, extract_suprasegmental_features_cached

_warm_sample = df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].sample(
    n=min(20, len(df_m6)), random_state=config.DEFAULT_SEED)
for _row in _warm_sample.itertuples(index=False):
    extract_segmental_features_cached(_row.Filepath)
    extract_suprasegmental_features_cached(_row.Filepath)

budget = ExperimentBudgetManager(models=[SEVERITY_MODEL_NAME],
                                 hard_cap_hours=config.PRIMARY_SEVERITY_BUDGET_HOURS,
                                 n_folds=15, name="severity_primary")
budget.benchmark(df_m6, task="severity")
budget.preflight()

In [ ]:
# COMPUTE BUDGET, STEP 2 -- confirm the locked epoch/patience budget and allocate the session's wall-clock...
assert cfg.patience == config.DEFAULT_PATIENCE and cfg.epochs == config.DEFAULT_EPOCHS
budget.allocate()
print_status(f"Compute budget locked: patience={cfg.patience}, epoch ceiling={cfg.epochs}, "
            f"hard cap={config.PRIMARY_SEVERITY_BUDGET_HOURS}h", ok=True)

In [ ]:
check_frozen_config_guard(final_config)
frozen_path = write_frozen_config(final_config)
print_status(f"Configuration frozen -> {frozen_path}", ok=True)
print_status("Configuration locked for the one real training run.", ok=True)

In [ ]:
final_run_cfg = TrainingConfig(
    task="severity", model=SEVERITY_MODEL_NAME, run_name=RUN_NAME,
    severity_protocol="full_loso", epochs=cfg.epochs, patience=cfg.patience,
    max_folds=None, limit_samples=None,
)

# The wall-clock compute budget (COMPUTE BUDGET, STEP 0-2 above) is enforced here, via deadline=.
print_header("REAL ONE-SHOT TRAINING RUN")
run_deadline = budget.deadline_for(SEVERITY_MODEL_NAME, allow_partial=True)
print_status(f"Compute-budget deadline wired in -- run_training will stop cleanly at a "
            f"fold boundary once the {config.PRIMARY_SEVERITY_BUDGET_HOURS}h session cap "
            "is reached.", ok=True)

summary, pooled_metrics = run_training(df_m6, final_run_cfg, deadline=run_deadline)

In [ ]:
metrics_dir = config.METRICS_DIR / RUN_NAME
per_fold_path = metrics_dir / f"{RUN_NAME}.per_fold.csv"

if per_fold_path.exists():
    per_fold_df = pd.read_csv(per_fold_path)
    display_cols = [c for c in ["fold", "epochs_completed", "test_loss", "accuracy",
                                "train_time_s"] if c in per_fold_df.columns]
    per_fold_df[display_cols]
else:
    per_fold_df = pd.DataFrame()
    print_status("No per-fold metrics yet for this run -- run training first.", ok=False)

In [ ]:
pooled_path = config.METRICS_DIR / RUN_NAME / "ALL_FOLDS_pooled.json"

if pooled_path.exists():
    with open(pooled_path) as f:
        pooled = json.load(f)
    for k, v in pooled.items():
        print_kv(k, v)
else:
    print_status("No pooled metrics yet for this run.", ok=False)

In [ ]:
try:
    gate_df = load_run_gate_weights(RUN_NAME)
    gate_summary = summarize_gate_values(gate_df)
    plot_gate_distribution(gate_df, show=True)
    gate_summary
except FileNotFoundError:
    print_status("No gate weights saved yet for this run.", ok=False)

In [ ]:
if not per_fold_df.empty and "complementarity_penalty" in per_fold_df.columns:
    print_kv("Complementarity penalty (mean +/- std)",
             f"{per_fold_df['complementarity_penalty'].mean():.5f} +/- "
             f"{per_fold_df['complementarity_penalty'].std():.5f}")
    print_kv("lambda_comp (frozen)", config.LAMBDA_COMP)
else:
    print_status("No complementarity-penalty diagnostics saved yet for this run.", ok=False)

In [ ]:
if not per_fold_df.empty and "speaker_accuracy" in per_fold_df.columns:
    print_kv("Speaker-head accuracy (mean +/- std)",
             f"{per_fold_df['speaker_accuracy'].mean():.3f} +/- "
             f"{per_fold_df['speaker_accuracy'].std():.3f}")
    print_kv("lambda_speaker (frozen)", config.LAMBDA_SPEAKER)
    print_status("Lower speaker-head accuracy suggests the fused representation carries less "
                "speaker-identifying information -- a diagnostic, not proof of invariance.",
                ok=True)
else:
    print_status("No speaker-adversarial diagnostics saved yet for this run.", ok=False)

In [ ]:
print_header("FINAL MODEL CHECKPOINTS")

ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME
if ckpt_dir.exists():
    fold_ckpts = sorted(ckpt_dir.glob("*/best.pt"))
    print_kv("Checkpoints saved", f"{len(fold_ckpts)} fold(s) (one best.pt per LOSO held-out speaker)")
    print_kv("Checkpoint directory", ckpt_dir)
    for ckpt_path in fold_ckpts:
        size_mb = ckpt_path.stat().st_size / (1024 ** 2)
        print_kv(f"  {ckpt_path.parent.name}", f"{ckpt_path} ({size_mb:.1f} MB)")
    print_status(f"All {len(fold_ckpts)} fold checkpoints saved as .pt via torch.save "
                "(model_state, optimizer_state, scheduler_state, scaler_state, epoch, "
                "monitored_value) -- see src/training/checkpoint.py.", ok=True)
else:
    print_status("No checkpoints saved yet for this run -- run training first.", ok=False)